In [ ]:
!uv pip install transformers datasets tqdm accelerate

Using Python 3.12.13 environment at: /usr
Checked 4 packages in 351ms


In [ ]:
import torch
torch.cuda.is_available()

False

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# ! pip install transformers[torch] datasets tqdm accelerate --only-binary :all: -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


# Dependencies

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, pipeline, BertTokenizerFast
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
import time
import os
import json, pickle as pkl



# Loading Dataset

In [ ]:
! pip install kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Importing twitter sentiment dataset

In [ ]:
!kaggle datasets download -d kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
100% 80.9M/80.9M [00:00<00:00, 96.9MB/s]



In [ ]:
# extracting the compressed dataset

from zipfile import ZipFile
dataset = '/content/sentiment140.zip'

with ZipFile(dataset, 'r') as zip:
  zip.extractall()
  print('The dataset is extracted successfully')

The dataset is extracted successfully


In [ ]:
# df = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1', header = None)
df = pd.read_csv('/content/drive/MyDrive/Sentiment Analysis Project/datasets/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1', header = None)

df.columns = ['target', 'id', 'date', 'flag', 'user', 'text']

# Converting labels
df['target'] = df['target'].replace(4,1)

# Keeping only needed columns
df = df[['text', 'target']]


## Create 3 datasets

- Dataset 1: Small (50k per class → 100k total)
- Dataset 2: Medium (100k per class → 200k total) MAIN
- Dataset 3: Large (200k per class → 400k total)

In [ ]:
df_small = df.groupby('target').sample(50000, random_state=42).reset_index(drop=True)

In [ ]:
df_medium = df.groupby('target').sample(100000, random_state=42).reset_index(drop=True)

In [ ]:
df_large = df.groupby('target').sample(200000, random_state=42).reset_index(drop=True)

Sanity check

In [ ]:
print("Small:\n", df_small['target'].value_counts())
print("\nMedium:\n", df_medium['target'].value_counts())
print("\nLarge:\n", df_large['target'].value_counts())

Small:
 target
0    50000
1    50000
Name: count, dtype: int64

Medium:
 target
0    100000
1    100000
Name: count, dtype: int64

Large:
 target
0    200000
1    200000
Name: count, dtype: int64


In [ ]:
df_small.to_csv("sentiment_small.csv", index=False)
df_medium.to_csv("sentiment_medium.csv", index=False)
df_large.to_csv("sentiment_large.csv", index=False)

# df_small.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_small.csv", index=False)
# df_medium.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_medium.csv", index=False)
# df_large.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_large.csv", index=False)

# REUSABLE BERT EXPERIMENT PIPELINE

## Metrics function

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=1)

  precision, recall,f1, _ = precision_recall_fscore_support(labels, preds, average = 'binary')
  acc = accuracy_score(labels, preds)

  probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
  roc = roc_auc_score(labels, probs)

  return {
      "accuracy": acc,
      "f1": f1,
      "roc_auc": roc,
      "precision": precision,
      "recall": recall
  }




MAIN reusable function

In [ ]:
def run_experiment(
    df,
    dataset_name="dataset",
    save_dir="/content/drive/MyDrive/Sentiment Analysis Project/models"
):
    import os, time, json, pickle as pkl
    import torch
    from sklearn.model_selection import train_test_split
    from transformers import (
        DistilBertTokenizerFast,
        DistilBertForSequenceClassification,
        TrainingArguments,
        Trainer
    )

    print(f"Running experiment on {dataset_name}")


    # Save directory

    exp_path = os.path.join(save_dir, dataset_name)
    os.makedirs(exp_path, exist_ok=True)


    # Split

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['text'].tolist(),
        df['target'].tolist(),
        test_size=0.2,
        random_state=42,
        stratify=df['target']
    )

    with open(os.path.join(exp_path, "data_split_info.json"), "w") as f:
        json.dump({
            "train_size": len(train_texts),
            "val_size": len(val_texts)
        }, f, indent=4)


    # Tokenizer

    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

    train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
    val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)


    # Dataset

    class SentimentDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels

        def __getitem__(self, idx):
            item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
            item["labels"] = torch.tensor(self.labels[idx])
            return item

        def __len__(self):
            return len(self.labels)

    train_dataset = SentimentDataset(train_encodings, train_labels)
    val_dataset = SentimentDataset(val_encodings, val_labels)


    # Model

    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )


    # Training Args (FAST MODE FIXED)

    training_args = TrainingArguments(
        output_dir=os.path.join(exp_path, "checkpoints"),

        num_train_epochs=2,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=2,

        learning_rate=2e-5,

        # FIXED
        eval_strategy="no",
        save_strategy="no",
        load_best_model_at_end=False,

        fp16=True,
        optim="adamw_torch",

        dataloader_num_workers=4,
        dataloader_pin_memory=True,

        logging_steps=200,
        report_to="tensorboard"
    )


    # Trainer

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )


    # Train

    start = time.time()
    trainer.train()
    end = time.time()

    trainer.save_state()


    # Evaluate

    results = trainer.evaluate()

    final_results = {
        "dataset": dataset_name,
        "accuracy": results.get("eval_accuracy"),
        "f1": results.get("eval_f1"),
        "roc_auc": results.get("eval_roc_auc"),
        "training_time_sec": end - start
    }


    # Save everything

    trainer.save_model(exp_path)
    tokenizer.save_pretrained(exp_path)

    with open(os.path.join(exp_path, "results.json"), "w") as f:
        json.dump(final_results, f, indent=4)

    with open(os.path.join(exp_path, "results.pkl"), "wb") as f:
        pkl.dump(final_results, f)

    with open(os.path.join(exp_path, "training_args.json"), "w") as f:
        json.dump(training_args.to_dict(), f, indent=4)

    label_map = {0: "negative", 1: "positive"}
    with open(os.path.join(exp_path, "label_map.json"), "w") as f:
        json.dump(label_map, f, indent=4)

    print(f" Experiment saved at: {exp_path}")

    return final_results

In [ ]:
# def run_experiment(
#   df,
#   dataset_name="dataset",
#   save_dir="/content/drive/MyDrive/Sentiment Analysis Project/models"
# ):

#   print(f"Running experiment on {dataset_name}")


#   # Create save directory
#   exp_path = os.path.join(save_dir, dataset_name)
#   os.makedirs(exp_path, exist_ok=True)


#   # 1. Split

#   train_texts, val_texts, train_labels, val_labels = train_test_split(
#       df['text'].tolist(),
#       df['target'].tolist(),
#       test_size=0.2,
#       random_state=42,
#       stratify=df['target']
#   )

#   # Save split info (VERY useful)
#   with open(os.path.join(exp_path, "data_split_info.json"), "w") as f:
#       json.dump({
#           "train_size": len(train_texts),
#           "val_size": len(val_texts)
#       }, f, indent=4)


#   # 2. Tokenizer

#   # tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

#   tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

#   train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
#   val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)


#   # 3. Dataset

#   class SentimentDataset(torch.utils.data.Dataset):
#       def __init__(self, encodings, labels):
#           self.encodings = encodings
#           self.labels = labels

#       def __getitem__(self, idx):
#           item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
#           item['labels'] = torch.tensor(self.labels[idx])
#           return item

#       def __len__(self):
#           return len(self.labels)

#   train_dataset = SentimentDataset(train_encodings, train_labels)
#   val_dataset = SentimentDataset(val_encodings, val_labels)


#   # 4. Model

#   model = BertForSequenceClassification.from_pretrained(
#       'bert-base-uncased',
#       num_labels=2
#   )


#   # 5. Training args

#   training_args = TrainingArguments(
#       output_dir=os.path.join(exp_path, "checkpoints"),
#       num_train_epochs=2,
#       per_device_train_batch_size=32,
#       per_device_eval_batch_size=32,
#       gradient_accumulation_steps=2,
#       learning_rate=2e-5,

#       # eval_strategy="steps",
#       eval_strategy="no",

#       eval_steps=2000,
#       save_strategy="epoch",   # enable checkpoint saving
#       save_steps=2000,

#       fp16=True,
#       optim="adamw_torch_fused",

#       dataloader_num_workers=4,
#       dataloader_pin_memory=True,

#       logging_steps=200,
#       logging_dir=os.path.join(exp_path, "logs"),

#       load_best_model_at_end=True
#   )


#   # 6. Trainer

#   trainer = Trainer(
#       model=model,
#       args=training_args,
#       train_dataset=train_dataset,
#       eval_dataset=val_dataset,
#       compute_metrics=compute_metrics
#   )


#   # 7. Train

#   start = time.time()
#   trainer.train()
#   end = time.time()

#   # Save trainer state (resume capability)
#   trainer.save_state()


#   # 8. Evaluate

#   results = trainer.evaluate()

#   final_results = {
#       "dataset": dataset_name,
#       "accuracy": results.get("eval_accuracy"),
#       "f1": results.get("eval_f1"),
#       "roc_auc": results.get("eval_roc_auc"),
#       "training_time_sec": end - start
#   }


#   # 9. SAVE EVERYTHING


#   # Model + tokenizer
#   trainer.save_model(exp_path)
#   tokenizer.save_pretrained(exp_path)

#   # Results (JSON + PKL)
#   with open(os.path.join(exp_path, "results.json"), "w") as f:
#       json.dump(final_results, f, indent=4)

#   with open(os.path.join(exp_path, "results.pkl"), "wb") as f:
#       pkl.dump(final_results, f)

#   # Training args
#   with open(os.path.join(exp_path, "training_args.json"), "w") as f:
#       json.dump(training_args.to_dict(), f, indent=4)

#   # Label mapping (important for inference)
#   label_map = {0: "negative", 1: "positive"}
#   with open(os.path.join(exp_path, "label_map.json"), "w") as f:
#       json.dump(label_map, f, indent=4)

#   print(f" Experiment saved at: {exp_path}")

#   return final_results

In [ ]:
# def run_experiment(df, dataset_name="dataset"):
#   print(f"Running experiment on {dataset_name}")

#   # 1. Split
#   train_texts, val_texts, train_labels, val_labels = train_test_split(
#       df['text'].tolist(), df['target'].tolist(), test_size=0.2, random_state=42, stratify=df['target']
#     )

#   # 2. Tokenizer
#   tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

#   train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
#   val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

#   # 3. Dataset class
#   class SentimentDataset(torch.utils.data.Dataset):
#     def __init__(self, encodings, labels):
#       self.encodings = encodings
#       self.labels = labels

#     def __getitem__(self, idx):
#       item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
#       item['labels'] = torch.tensor(self.labels[idx])
#       return item

#     def __len__(self):
#       return len(self.labels)


#   train_dataset = SentimentDataset(train_encodings, train_labels)
#   val_dataset = SentimentDataset(val_encodings, val_labels)

#   # 4. Model
#   model = BertForSequenceClassification.from_pretrained(
#         'bert-base-uncased',
#         num_labels=2
#   )

#   # 5. Training args
#   # training_args = TrainingArguments(
#   #     output_dir=f"./results_{dataset_name}",
#   #     num_train_epochs=2,
#   #     per_device_train_batch_size=8,
#   #     per_device_eval_batch_size=8,
#   #     learning_rate=2e-5,
#   #     eval_strategy="epoch",
#   #     save_strategy="epoch",
#   #     load_best_model_at_end=True,
#   #     fp16=True,
#   #     logging_dir=f"./logs_{dataset_name}"
#   # )
#   training_args = TrainingArguments(
#   output_dir=f"./results_{dataset_name}",
#   num_train_epochs=2,
#   per_device_train_batch_size=16,
#   per_device_eval_batch_size=16,
#   gradient_accumulation_steps=2,
#   learning_rate=2e-5,

#   evaluation_strategy="steps",
#   eval_steps=2000,
#   save_strategy="no",

#   fp16=True,
#   optim="adamw_torch_fused",

#   dataloader_num_workers=2,
#   dataloader_pin_memory=True,

#   logging_steps=100,
#   logging_dir=f"./logs_{dataset_name}"
#   )

#   # 6. Trainer
#   trainer = Trainer(
#       model=model,
#       args=training_args,
#       train_dataset=train_dataset,
#       eval_dataset=val_dataset,
#       compute_metrics=compute_metrics
#   )

#   # 7. Train
#   start = time.time()
#   trainer.train()
#   end = time.time()

#   # 8. Evaluate
#   results = trainer.evaluate()

#   return {
#       "dataset": dataset_name,
#       "accuracy": results["eval_accuracy"],
#       "f1": results["eval_f1"],
#       "roc_auc": results["eval_roc_auc"],
#       "training_time_sec": end - start
#   }

# Run all experiments

In [ ]:
import transformers
print(transformers.__version__)

5.0.0


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
results_small = run_experiment(df_small, "df_small")

results_small

Running experiment on df_small


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

Step,Training Loss,Validation Loss,Accuracy,F1,Roc Auc,Precision,Recall
2000,0.713630,0.369119,0.838050,0.831662,0.920222,0.865815,0.800100
4000,0.551956,0.381035,0.845650,0.848535,0.924509,0.832964,0.864700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Experiment saved at: /content/drive/MyDrive/Sentiment Analysis Project/models/df_small


{'dataset': 'df_small',
 'accuracy': 0.8377,
 'f1': 0.8314641744548287,
 'roc_auc': 0.9202532600000001,
 'training_time_sec': 1319.468113899231}

In [ ]:
!nvidia-smi

Tue Apr 28 10:56:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             33W /   70W |    3019MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# results_medium = run_experiment(df_medium, "medium_100k")
results_medium = run_experiment(df_medium, "df_medium")

results_medium

In [ ]:
results_large = run_experiment(df_large, "large_200k")
results_large

# Compare results

In [ ]:
results_df = pd.DataFrame([
    results_small,
    results_medium,
    results_large
])

results_df